# GJ-NR-001: Exact Main Title Match

Start Docker Spark cluster → Run Step 3 normalization → Verify record → Shut down cluster

## 1. Start Docker Spark Cluster

In [ ]:
import subprocess, os, time

# gilJOBi root directory (2 levels up from notebook location)
GILJOBI_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
print(f"gilJOBi directory: {GILJOBI_DIR}")

# Start Spark + Postgres only (Airflow/Redis/Livy not needed for NR testing)
result = subprocess.run(
    ['docker', 'compose', 'up', '-d', 'spark-master', 'spark-worker-1', 'spark-worker-2', 'postgres'],
    cwd=GILJOBI_DIR, capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

# Wait for containers to stabilize
time.sleep(10)

# Verify container status
ps = subprocess.run(['docker', 'ps', '--format', 'table {{.Names}}\t{{.Status}}'], capture_output=True, text=True)
print(ps.stdout)

## 2. Run Step 3 Normalization

In [ ]:
result = subprocess.run(
    ['docker', 'exec', 'spark-master',
     '/opt/spark/bin/spark-submit',
     '/opt/spark/notebooks/step3_rule_based_primary_tag.py'],
    capture_output=True, text=True, timeout=300
)

# Print Matching Summary
output = result.stdout + result.stderr
for line in output.split('\n'):
    if any(kw in line for kw in ['Matching Summary', 'exact_main', 'exact_sub', 'partial', 'no_match', 'TOTAL MATCHED', 'STEP 3', 'Loaded', 'Saved']):
        print(line)

## 3. Verify Record — `'software developer'`

In [ ]:
import pandas as pd

# Read step3 output parquet (mounted on host via docker-compose volume)
parquet_path = os.path.join(GILJOBI_DIR, 'data', 'processed', 'step3_with_primary_tag')
df = pd.read_parquet(parquet_path)

print(f"Total records: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print()

# Filter for 'software developer' records
target = df[df['seniority_removed_title'].str.lower().str.strip() == 'software developer']
print(f"Records matching 'software developer': {len(target)}")
target[['title', 'seniority', 'seniority_removed_title', 'primary_tag', 'match_type']]

## 4. Shut Down Docker Cluster

In [ ]:
result = subprocess.run(
    ['docker', 'compose', 'down'],
    cwd=GILJOBI_DIR, capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)